> **Deprecated**
>
> This notebook has been consolidated into `secondary_documentation_notebook.ipynb`.
> Please refer to that notebook instead.
> 
> Section: **Geographic Descriptions of Representative Positions** (Section 8 in `secondary_documentation_notebook.ipynb`).

In [1]:
import pickle
with open('positions_2026-04-23.pkl', 'rb') as f:
    positions_ordered = pickle.load(f)


In [ ]:
import numpy as np
import pandas as pd
import geonamescache

# --- KONFIGURATION ---
CITY_MIN_POPULATION = 150000

# --- EINGABE ---
clusters =positions_ordered

# --- HILFSFUNKTIONEN ---

def haversine_np(lon1, lat1, lon2, lat2):
    """Entfernung in km (Eingabe: Radians)."""
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))


def bearing_deg(lat1, lon1, lat2, lon2):
    """Kurswinkel in Grad (0° = Nord, im Uhrzeigersinn)."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360


def bearing_to_direction(deg):
    """Kurswinkel → Himmelsrichtung (Achtelkreise)."""
    directions = ["N", "NO", "O", "SO", "S", "SW", "W", "NW"]
    idx = round((deg-180) / 45) % 8
    return directions[idx]


def round_half(value):
    """Rundet auf 0,5 km."""
    return round(value * 2) / 2


def get_nearest_city(point_lat, point_lon, city_df, city_coords_rad):
    """Nächste Stadt + Entfernung in km."""
    p_lat = np.radians(point_lat)
    p_lon = np.radians(point_lon)
    dists = haversine_np(p_lon, p_lat, city_coords_rad[:, 1], city_coords_rad[:, 0])
    idx = np.argmin(dists)
    city = city_df.iloc[idx]
    return city["name"], city["lat"], city["lon"], city["country"], dists[idx]


def format_label(dist_km, direction, city_name):
    """Erstellt z.B. '15,5 km nördlich von Calais'."""
    direction_text = {
        "N": "nördlich",
        "NO": "nordöstlich",
        "O": "östlich",
        "SO": "südöstlich",
        "S": "südlich",
        "SW": "südwestlich",
        "W": "westlich",
        "NW": "nordwestlich",
    }
    dist_str = f"{dist_km:g}".replace(".", ",")
    if dist_km == 0.0:
        return city_name
    return f"{dist_str} km {direction_text[direction]} von {city_name}"


# --- STÄDTE-DATENBANK ---
gc = geonamescache.GeonamesCache()
cities = gc.get_cities()
countries_raw = gc.get_countries()

# Ländernamen auf Deutsch (nur europäische Länder, die hier vorkommen können)
COUNTRY_DE = {
    "AD": "Andorra", "AL": "Albanien", "AT": "Österreich", "BA": "Bosnien-Herzegowina",
    "BE": "Belgien", "BG": "Bulgarien", "BY": "Weißrussland", "CH": "Schweiz",
    "CY": "Zypern", "CZ": "Tschechien", "DE": "Deutschland", "DK": "Dänemark",
    "EE": "Estland", "ES": "Spanien", "FI": "Finnland", "FR": "Frankreich",
    "GB": "Vereinigtes Königreich", "GR": "Griechenland", "HR": "Kroatien",
    "HU": "Ungarn", "IE": "Irland", "IS": "Island", "IT": "Italien",
    "LI": "Liechtenstein", "LT": "Litauen", "LU": "Luxemburg", "LV": "Lettland",
    "MC": "Monaco", "MD": "Moldau", "ME": "Montenegro", "MK": "Nordmazedonien",
    "MT": "Malta", "NL": "Niederlande", "NO": "Norwegen", "PL": "Polen",
    "PT": "Portugal", "RO": "Rumänien", "RS": "Serbien", "RU": "Russland",
    "SE": "Schweden", "SI": "Slowenien", "SK": "Slowakei", "SM": "San Marino",
    "TR": "Türkei", "UA": "Ukraine", "VA": "Vatikanstadt", "XK": "Kosovo",
}

european_cities = [
    {
        "name": c["name"],
        "lat": c["latitude"],
        "lon": c["longitude"],
        "country": COUNTRY_DE.get(c["countrycode"], countries_raw.get(c["countrycode"], {}).get("name", c["countrycode"])),
    }
    for c in cities.values()
    if c["timezone"] and c["timezone"].startswith("Europe/")
    and c["population"] > CITY_MIN_POPULATION
]
city_df = pd.DataFrame(european_cities)
city_coords_rad = np.radians(city_df[["lat", "lon"]].values)

# --- HAUPTSCHLEIFE ---
rows = []

for group_id, positions in clusters.items():
    for pos_idx, (lon, lat) in enumerate(positions):
        city_name, city_lat, city_lon, country, dist_km = get_nearest_city(
            lat, lon, city_df, city_coords_rad
        )
        dist_rounded = round_half(dist_km)
        deg = bearing_deg(lat, lon, city_lat, city_lon)
        direction = bearing_to_direction(deg)
        label = format_label(dist_rounded, direction, city_name) + f", {country}"

        rows.append({
            "gruppe":          group_id,
            "nr":              pos_idx,
            "lon":             round(lon, 6),
            "lat":             round(lat, 6),
            "stadt":           city_name,
            "land":            country,
            "entfernung_km":   dist_rounded,
            "richtung":        direction,
            "label":           label,
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

df.to_csv("output/tabular_and_text/cluster_locations.csv", index=False)
print("\nGespeichert als cluster_locations.csv")

 gruppe  nr       lon       lat             stadt                   land  entfernung_km richtung                                                        label
      0   0 12.779639 45.437292            Padova                Italien           70.0        O                            70 km östlich von Padova, Italien
      0   1 11.224555 54.614898              Kiel            Deutschland           77.5       NO                    77,5 km nordöstlich von Kiel, Deutschland
      0   2  0.372386 52.873527      Peterborough Vereinigtes Königreich           53.5       NO 53,5 km nordöstlich von Peterborough, Vereinigtes Königreich
      1   0 11.523492 54.561328           Rostock            Deutschland           66.0       NW                  66 km nordwestlich von Rostock, Deutschland
      1   1 -0.038627 52.502685      Peterborough Vereinigtes Königreich           16.0       SO    16 km südöstlich von Peterborough, Vereinigtes Königreich
      1   2 -1.278461 46.252154            Nantes   